# Create windows for CNN training

In [ ]:
import sys
sys.path.append('../../')

import matplotlib.pyplot as plt

from param_classes.workflows.input_output import InputDataParams
from workflows.run_util.io import collect_paths_from_table

input_params = InputDataParams()

## Data settings

Set raw input dir. You need to give the base path where the group directories can be found.

In [ ]:
input_params.raw_data.path ='E:\\Data\\BogiSejtek\\20251007\\data\\invivo'
input_params.raw_data.extension = 'mat'

Set the base path of the directory where csv files containing the spike features and categories can be found.
> It needs to be the same structure as the `raw_data` directory.

In [ ]:
input_params.spike_features.path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\SpikeFeatures'
input_params.spike_features.extension = 'csv'

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [ ]:
input_selector = {
    'mode': 'table',
    'table_path': 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\data_invivo_noisetest.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
output_path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\SpikeOcurrencePrediction\\01_training_data'

## Analysis parameters

In [ ]:
signal_window_size = 0.25 # seconds
signal_window_shift = 0.05 # seconds
signal_fs = 1000 # Hz
spike_window_size = 0.25
spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']

## Run

### Preparation

In [ ]:
signal_window_in_samples = int(signal_window_size * signal_fs)
signal_window_shift_in_samples = int(signal_window_shift * signal_fs)

In [ ]:
input_files = collect_paths_from_table(input_params, input_selector)

In [ ]:
for input_file in input_files:
    if input_file.name != 'B18_d3t1c1':
        continue
    data = input_file.data.load()
    signal = data.get_signal('wideband', fs=signal_fs)
    times, values = signal.times, signal.values
    signal_segments = []
    spike_ocurrences = []
    for win_start_i in range(0, len(values) - signal_window_in_samples, int(signal_window_shift * signal_fs)):
        win_end_i = win_start_i + signal_window_in_samples
        signal_segment = values[win_start_i:win_end_i]
        spike_window = (
            times[win_start_i] + signal_window_size,
            times[win_start_i] + signal_window_size + spike_window_size
        )
        signal_segments.append(signal_segment)